## TinyLoRA に MoE を使う

TinyLoRA でランダム行列が使われることに納得がいかず、ランダム行列を遺伝的アルゴリズム(GA)で学習する方向を ChatGPT さんに試してもらったが、良い結果が出せなかった。良い結果を出すには、遺伝的アルゴリズムではなく MoE (Mixture of Experts) を使う方向だというのが ChatGPT さんの意見である。MoE なら改善されることを試すミニコードがこのコードである。ChatGPT 5.3 (?) さんが作ったそのままのコードである。

解説は↓。

\[cocolog:95917075](2026年3月)  
《TinyLoRA は普通の LoRA が和なのに対し、積の LoRA のような感じになるようだ。極小パラメータの TinyLoRA は生物の「ファインチューニング」の効率の良さとつながりがあるという直感から興味を持ったのだった。 - JRF のひとこと》  
http://jrf.cocolog-nifty.com/statuses/2026/03/post-b22b37.html


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import random

device = torch.device("cpu")

# -----------------------------
# problem setup
# -----------------------------

in_f = 32
out_f = 32
r = 6

dataset_size = 256
batch_size = 64
steps = 150
lr = 1e-2

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# base weight
W_base = torch.randn(out_f, in_f) * 0.5

U_full, S_full, Vh_full = torch.linalg.svd(W_base, full_matrices=False)

U = U_full[:, :r]
S = S_full[:r]
V = Vh_full[:r, :].T

U_scaled = U * S.unsqueeze(0)

# synthetic target adapter
rng = np.random.RandomState(1)

R_target = torch.tensor(rng.normal(size=(r, r)), dtype=torch.float32)
R_target = R_target / (R_target.norm() + 1e-12)

adapter_full = U_scaled @ R_target @ V.T
W_true = W_base + adapter_full

X = torch.randn(dataset_size, in_f)
Y = F.linear(X, W_true)

# -----------------------------
# random basis generator
# -----------------------------

def make_random_A(r, u, seed):

    rng = np.random.RandomState(seed)

    A = []

    for _ in range(u):

        a = torch.tensor(
            rng.normal(size=(r, r)),
            dtype=torch.float32
        )

        a = a / (a.reshape(-1).norm() + 1e-12)

        A.append(a)

    return torch.stack(A)

# -----------------------------
# Mixture-of-subspaces TinyLoRA
# -----------------------------

class TinyLoRA_MoS(torch.nn.Module):

    def __init__(self, r, u, K):

        super().__init__()

        self.r = r
        self.u = u
        self.K = K

        # random subspaces
        A = []

        for k in range(K):
            A.append(make_random_A(r, u, seed=1000+k))

        self.A = torch.stack(A)  # (K,u,r,r)

        # learnable parameters
        self.v = torch.nn.Parameter(
            torch.randn(K, u) * 1e-3
        )

        self.w = torch.nn.Parameter(
            torch.zeros(K)
        )

    def compute_R(self):

        weights = torch.softmax(self.w, dim=0)

        R = 0

        for k in range(self.K):

            Ak = self.A[k]

            Rk = torch.einsum(
                "uij,u->ij",
                Ak,
                self.v[k]
            )

            R = R + weights[k] * Rk

        return R


# -----------------------------
# training
# -----------------------------

def train_model(u=4, K=4):

    model = TinyLoRA_MoS(r=r, u=u, K=K)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    for step in range(steps):

        idx = np.random.choice(
            dataset_size,
            batch_size,
            replace=False
        )

        xb = X[idx]
        yb = Y[idx]

        R = model.compute_R()

        W = W_base + U_scaled @ R @ V.T

        out = F.linear(xb, W)

        loss = F.mse_loss(out, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

    with torch.no_grad():

        R = model.compute_R()

        W = W_base + U_scaled @ R @ V.T

        out = F.linear(X, W)

        loss = F.mse_loss(out, Y)

    return loss.item()


# -----------------------------
# experiment
# -----------------------------

for K in [1,2,4,8]:

    loss = train_model(u=4, K=K)

    print("K =",K,"loss =",loss)

K = 1 loss = 0.7558706998825073
K = 2 loss = 0.6951342225074768
K = 4 loss = 0.40621843934059143
K = 8 loss = 0.22985006868839264
